**TASK 1 : Load and Inspect**

In [77]:
# AI-assisted: Asked ChatGPT for dataset inspection structure.

import pandas as pd
import numpy as np

#load csv file
df = pd.read_csv('bangalore_tech_salaries.csv')

df

,Employee_ID,Role,years_exp,Current_CTC,Previous_CTC,Company,company_TYPE,Skills,Location,Education_Tier,Joining_Year,Work_Mode
0,BLR0065,DS,6,49.4,32.4 LPA,Ola,Unicorn,"JIRA, JavaScript, Spring Boot, Figma, NumPy, A...",HSR Layout,Tier-1,2022,Remote
1,BLR0080,DevOps,0,9.7,NaN,Walmart Labs,MNC,"Tableau, Python, Figma, JavaScript, TensorFlow",Electronic City,2,2024,Remote
2,BLR0166,SDE Backend,6,"3,360,000",24.2 LPA,Postman,Mid-size,"Agile, Excel, GCP, MongoDB, Java, ML",Whitefield,Tier-1,2023,Hybrid
3,BLR0219,SDE Frontend,8,41.4 LPA,30.3,Amazon India,MNC,"PyTorch, Redis, System Design",MG Road,Tier-2,2022,Work from Office
4,BLR0491,Product Lead,5,"3,640,000",30.6 LPA,QuickPay AI,early-stage,"Tableau, Deep Learning, Agile",Whitefield,Tier-2,2023,Onsite
...,...,...,...,...,...,...,...,...,...,...,...,...
1010,BLR0824,Product Designer,2,17.6,12.4,Zoho,Mid-size,"React, SQL, ML, Deep Learning, Java",Jayanagar,Tier-2,2022,WFH
1011,BLR0792,Product Lead,4,₹40.2 LPA,26.6,HCL,MNC,"Excel, Java, Deep Learning, Python, JavaScript",Whitefield,Tier 2,2021,Work from Office
1012,BLR0209,BI Analyst,1,8.6 LPA,6.0 LPA,EdMint,Early-stage,"PyTorch, System Design, MongoDB, JIRA, PowerBI...",NaN,2,2024,Hybrid
1013,BLR0788,PM,2,27.6,17.4,Zomato,Unicorn,"Azure, Kafka, Excel",Bellandur,Tier-3,2023,Remote


In [35]:
# Basic information
print("Shape:", df.shape)
print("\nInformation")
print(df.info())

print("\nData Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# Dataset Summary
print("\n========== DATASET SUMMARY ==========")
print(f"Total Rows : {df.shape[0]}")
print(f"Total Columns : {df.shape[1]}")
print(f"Duplicate Rows : {df.duplicated().sum()}")

print("\nColumns with Missing Values")
print(df.isnull().sum().sort_values(ascending=False))


Shape: (1015, 12)

Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
None

Data Types
Employee_ID       object
Role              object
years_exp          int64
Current_CTC       object
Previous_CTC      object
Company           object
company_TYPE      object
Sk

**TASK 2 : Clean Dataset**

Column Renaming & Standardizing Categorical Columns

In [81]:
# 1. Clean Column Names (Strip trailing spaces like 'Role ' and map to snake_case)
df.columns = df.columns.str.strip().str.lower()

# 2. String Cleanup: Strip whitespaces from string columns to prevent matching errors
string_cols = ['role', 'company', 'company_type', 'education_tier', 'work_mode', 'location']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# 3. Standardize Company Type
company_type_map = {
    'unicorn': 'Unicorn', 'UNICORN': 'Unicorn', 'Unicorn': 'Unicorn',
    'mnc': 'MNC', 'MNC': 'MNC',
    'mid-size': 'Mid-size', 'MID-SIZE': 'Mid-size', 'Mid-size': 'Mid-size',
    'early-stage': 'Early-stage', 'Early-stage': 'Early-stage', 'EARLY-STAGE': 'Early-stage'
}
df['company_type'] = df['company_type'].map(company_type_map)

# 4. Standardize Education Tier
education_tier_map = {
    '1': 'Tier-1', 'T1': 'Tier-1', 'Tier 1': 'Tier-1', 'Tier-1': 'Tier-1',
    '2': 'Tier-2', 'T2': 'Tier-2', 'Tier 2': 'Tier-2', 'Tier-2': 'Tier-2',
    '3': 'Tier-3', 'T3': 'Tier-3', 'Tier 3': 'Tier-3', 'Tier-3': 'Tier-3'
}
df['education_tier'] = df['education_tier'].map(education_tier_map)

print("Standardized Education Tiers:", df['education_tier'].unique())
print("Standardized Company Types:", df['company_type'].unique())

Standardized Education Tiers: ['Tier-1' 'Tier-2' 'Tier-3']
Standardized Company Types: ['Unicorn' 'MNC' 'Mid-size' 'Early-stage']


Standardizing Job Roles

In [83]:
role_mapping = {
    # Data Science & Analytics
    'DS': 'Data Scientist', 'Data Scientist': 'Data Scientist', 'Data Science Engineer': 'Data Scientist',
    'ML Engineer': 'Data Scientist',
    'DA': 'Data Analyst', 'Data Analyst': 'Data Analyst', 'BA': 'Business Analyst',
    'Business Analyst': 'Business Analyst', 'BI Analyst': 'Business Analyst',
    'Business Systems Analyst': 'Business Analyst', 'Analytics Engineer': 'Data Analyst',

    # Software Engineering (Backend)
    'SDE Backend': 'SDE Backend', 'Backend Dev': 'SDE Backend', 'Backend Developer': 'SDE Backend',
    'Backend Engineer': 'SDE Backend', 'SDE-Backend': 'SDE Backend', 'BE': 'SDE Backend',

    # Software Engineering (Frontend)
    'SDE Frontend': 'SDE Frontend', 'SDE-Frontend': 'SDE Frontend', 'Frontend Dev': 'SDE Frontend',
    'Frontend Engineer': 'SDE Frontend', 'Frontend Developer': 'SDE Frontend', 'FE': 'SDE Frontend',

    # Software Engineering (Full Stack)
    'Full Stack Engineer': 'SDE Full-Stack', 'FullStack': 'SDE Full-Stack',
    'SDE FS': 'SDE Full-Stack', 'Fullstack Dev': 'SDE Full-Stack', 'SDE Full-Stack': 'SDE Full-Stack',

    # Product & Management
    'Product Lead': 'Product Manager', 'Product Manager': 'Product Manager', 'Sr PM': 'Product Manager',
    'PM': 'Product Manager',

    # DevOps & Infrastructure
    'DevOps': 'DevOps Engineer', 'DevOps Engineer': 'DevOps Engineer',
    'Site Reliability Engineer': 'DevOps Engineer', 'SRE': 'DevOps Engineer', 'Infra Engineer': 'DevOps Engineer',

    # Design (Unified with UI/UX fallback)
    'UI/UX': 'UI/UX Designer', 'UI Designer': 'UI/UX Designer', 'Designer': 'UI/UX Designer',
    'Product Designer': 'UI/UX Designer', 'UX Designer': 'UI/UX Designer'
}

df['role'] = df['role'].map(role_mapping)
print("Standardized Roles Summary:\n", df['role'].value_counts())

Standardized Roles Summary:
 role
Business Analyst    140
Data Scientist      128
DevOps Engineer     127
UI/UX Designer      116
SDE Backend         116
Product Manager     113
SDE Frontend        105
SDE Full-Stack       97
Data Analyst         73
Name: count, dtype: int64


Parsing Financial CTC Columns:
The 'Current_CTC' and 'Previous_CTC' columns have mixed data types, including string representations with 'LPA', 'Crore', and commas. I will create a function to clean these values and convert them into a consistent numerical format (in lakhs)

In [84]:
# We strip common symbols (₹, commas, spaces, 'LPA') and convert raw numbers > 1000 to Lakhs.

def clean_ctc_no_regex(ctc_val):
    """Parses a messy CTC value to a float in LPA using basic string manipulations."""
    # Handle missing values safely
    if pd.isna(ctc_val):
        return np.nan

    # Convert to standard lowercase string and strip outer spaces
    s = str(ctc_val).strip().lower()

    # Remove specific characters and currency units
    s = s.replace('₹', '')
    s = s.replace(',', '')
    s = s.replace('lpa', '')
    s = s.replace(' ', '')

    try:
        val = float(s)
        # If the value is a raw large figure (e.g., 3360000), scale it down to LPA
        if val > 1000:
            return round(val / 100000, 2)
        return round(val, 2)
    except ValueError:
        # Fallback if any unexpected characters remain
        return np.nan

# Apply the custom clean function to the target dataframe columns
df['current_ctc'] = df['current_ctc'].apply(clean_ctc_no_regex)
df['previous_ctc'] = df['previous_ctc'].apply(clean_ctc_no_regex)

# Quick sanity verification check on results
print("Sample of parsed Current CTC (LPA):")
print(df[['current_ctc', 'previous_ctc']].head(5))

Sample of parsed Current CTC (LPA):
   current_ctc  previous_ctc
0         49.4          32.4
1          9.7           NaN
2         33.6          24.2
3         41.4          30.3
4         36.4          30.6


Duplicate Handling & Missing Value Treatment

In [87]:
# Drop rows where current_ctc or role couldn't be parsed
df = df.dropna(subset=['current_ctc', 'role'])

# Drop row duplicates if exist
df = df.drop_duplicates()

# Fill uncritical missing text fields with 'Not Disclosed' or 'Unknown'
df['skills'] = df['skills'].fillna('Not Disclosed')
df['location'] = df['location'].fillna('Unknown')

print("Post-cleaning DataFrame Info:")
print(df.info())
print(df.head())

Post-cleaning DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   employee_id     1000 non-null   object 
 1   role            1000 non-null   object 
 2   years_exp       1000 non-null   int64  
 3   current_ctc     1000 non-null   float64
 4   previous_ctc    806 non-null    float64
 5   company         1000 non-null   object 
 6   company_type    1000 non-null   object 
 7   skills          1000 non-null   object 
 8   location        1000 non-null   object 
 9   education_tier  1000 non-null   object 
 10  joining_year    1000 non-null   int64  
 11  work_mode       1000 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 101.6+ KB
None
  employee_id             role  years_exp  current_ctc  previous_ctc  \
0     BLR0065   Data Scientist          6         49.4          32.4   
1     BLR0080  DevOps Eng

**TASK 3**

Question 3.1: CTC Distribution by Role

In [88]:
# Group by role and get basic salary stats (Median, Mean, Min, Max)
q31_result = df.groupby('role')['current_ctc'].agg(['median', 'mean', 'min', 'max'])

# Sort roles so the highest median salary is on top
q31_result = q31_result.sort_values(by='median', ascending=False)

print("--- Q3.1: SALARY DISTRIBUTIONS ---")
print(q31_result)

--- Q3.1: SALARY DISTRIBUTIONS ---
                  median       mean   min   max
role                                           
Product Manager    31.30  34.131250  10.8  80.1
Data Scientist     24.70  27.579200  10.0  75.6
SDE Full-Stack     22.35  25.409375   8.9  71.7
DevOps Engineer    22.10  23.267717   8.8  60.3
SDE Frontend       21.20  22.193333   6.7  84.4
SDE Backend        21.10  23.284211   7.9  55.1
UI/UX Designer     18.90  20.968142   6.2  63.3
Business Analyst   18.30  20.825000   5.2  52.7
Data Analyst       16.90  18.298611   5.6  43.4


Question 3.2: Experience Curve for SDE Backend

In [89]:
# 1. Filter out only SDE Backend employees
sde_backend = df[df['role'] == 'SDE Backend'].copy()

# 2. Divide years of experience into 4 simple buckets
sde_backend['exp_bucket'] = pd.cut(sde_backend['years_exp'], bins=[-1, 1, 3, 5, 100], labels=['0-1', '2-3', '4-5', '6+'])

# 3. Find the median salary for each experience bucket
q32_result = sde_backend.groupby('exp_bucket', observed=False)['current_ctc'].median()

print("--- Q3.2: SDE BACKEND EXPERIENCE TRAJECTORY ---")
print(q32_result)

--- Q3.2: SDE BACKEND EXPERIENCE TRAJECTORY ---
exp_bucket
0-1    11.65
2-3    20.00
4-5    25.85
6+     40.40
Name: current_ctc, dtype: float64


Question 3.3: Skill Premium for SDEs

In [90]:
# 1. Select all Software Engineers (Backend, Frontend, Full-Stack)
sde_all = df[df['role'].str.contains('SDE|Full-Stack', case=False, na=False)].copy()

print("--- Q3.3: STANDALONE SKILL PREMIUM FOR SDEs ---")

# 2. Manually check the median for each skill one by one
for skill in ['System Design', 'Kubernetes', 'ML', 'AWS']:
    # Check who has the skill and who does not
    has_skill = sde_all[sde_all['skills'].str.contains(skill, case=False)]
    no_skill = sde_all[~sde_all['skills'].str.contains(skill, case=False)]

    # Calculate medians
    median_with = has_skill['current_ctc'].median()
    median_without = no_skill['current_ctc'].median()
    premium = median_with - median_without

    print(f"{skill} -> With: {median_with:.1f} LPA | Without: {median_without:.1f} LPA | Premium: +{premium:.1f} LPA")

--- Q3.3: STANDALONE SKILL PREMIUM FOR SDEs ---
System Design -> With: 25.3 LPA | Without: 21.0 LPA | Premium: +4.3 LPA
Kubernetes -> With: 23.2 LPA | Without: 21.5 LPA | Premium: +1.7 LPA
ML -> With: 23.9 LPA | Without: 21.3 LPA | Premium: +2.6 LPA
AWS -> With: 20.9 LPA | Without: 21.9 LPA | Premium: +-0.9 LPA


Question 3.4: Company-Type Premium

In [91]:
# Use our sde_backend filter from Q3.2 and group by company type
q34_result = sde_backend.groupby('company_type')['current_ctc'].median()

# Sort to see who pays the most
q34_result = q34_result.sort_values(ascending=False)

print("--- Q3.4: SDE BACKEND SALARY BY COMPANY TYPE ---")
print(q34_result)

--- Q3.4: SDE BACKEND SALARY BY COMPANY TYPE ---
company_type
Unicorn        27.35
MNC            20.45
Mid-size       19.50
Early-stage    18.60
Name: current_ctc, dtype: float64


Question 3.5: Percentage Hike by Company Type

In [92]:
# 1. Drop rows that have missing previous salary data
switchers = df.dropna(subset=['previous_ctc']).copy()

# 2. Basic formula to calculate salary hike percentage
switchers['hike_pct'] = ((switchers['current_ctc'] - switchers['previous_ctc']) / switchers['previous_ctc']) * 100

# 3. Group by company type to find the median hike percentage
q35_result = switchers.groupby('company_type')['hike_pct'].median()

print("--- Q3.5: MEDIAN HIKE % BY COMPANY TYPE ---")
print(q35_result)

--- Q3.5: MEDIAN HIKE % BY COMPANY TYPE ---
company_type
Early-stage    37.201390
MNC            38.738739
Mid-size       39.522859
Unicorn        39.393939
Name: hike_pct, dtype: float64


**Task 4: Final Printed Report**

In [102]:
# Define variables needed for the report from previous analysis results
roles_sorted = q31_result.index.tolist()
exp_med = q32_result.to_dict()

# Calculate System Design skill premium components
sde_all = df[df['role'].str.contains('SDE|Full-Stack', case=False, na=False)].copy()
sd_with = sde_all[sde_all['skills'].str.contains('System Design', case=False)]['current_ctc'].median()
sd_without = sde_all[~sde_all['skills'].str.contains('System Design', case=False)]['current_ctc'].median()

comp_med = q34_result.to_dict()

# Calculate overall median hike percentage
switchers = df.dropna(subset=['previous_ctc']).copy()
switchers['hike_pct'] = ((switchers['current_ctc'] - switchers['previous_ctc']) / switchers['previous_ctc']) * 100
global_median_hike = switchers['hike_pct'].median()

print("=" * 62)
print("  BANGALORE TECH SALARY DECODER")
print("  Built by [Anushka Asawa] | The Unlox Academy | 2-Hour Live Project")
print("=" * 62)
print(f"  Dataset : {len(df)} Bengaluru tech professionals")
print("  Period  : 2024 employment snapshot")
print()

print("  ---- MEDIAN CTC BY ROLE (in LPA) ----")
for r in roles_sorted:
    med = df[df['role'] == r]['current_ctc'].median()
    bar_length = int(med / 1.5)  # Scale bar to fit width nicely
    bar = "█" * bar_length
    print(f"  {r:<18} {bar:<22} {med:>5.1f} LPA")
print()

print("  ---- SDE BACKEND EXPERIENCE TRAJECTORY ----")
print(f"  0-1 Years Exp (Entry)      : {exp_med.get('0-1', 0):>5.1f} LPA")
print(f"  2-3 Years Exp (Junior)     : {exp_med.get('2-3', 0):>5.1f} LPA")
print(f"  4-5 Years Exp (Mid-Level)  : {exp_med.get('4-5', 0):>5.1f} LPA")
print(f"  6+  Years Exp (Senior)     : {exp_med.get('6+', 0):>5.1f} LPA")
print()

print("  ---- SYSTEM DESIGN SKILL PREMIUM FOR SDEs ----")
print(f"  Engineers with System Design    : {sd_with:>5.1f} LPA")
print(f"  Engineers without System Design : {sd_without:>5.1f} LPA")
print(f"  Standalone Skill Premium Value  : +{(sd_with - sd_without):>4.1f} LPA")
print()

print("  ---- SDE BACKEND BY COMPANY ARCHITECTURE ----")
print(f"  Unicorn Tier               : {comp_med.get('Unicorn', 0):>5.1f} LPA")
print(f"  MNC Tier                   : {comp_med.get('MNC', 0):>5.1f} LPA")
print(f"  Mid-size Tier              : {comp_med.get('Mid-size', 0):>5.1f} LPA")
print(f"  Early-stage Tier           : {comp_med.get('Early-stage', 0):>5.1f} LPA")
print()

print("  ---- COMPENSATION SWITCH JUMP INCENTIVES ----")
print(f"  Overall Median Increment %  : {global_median_hike:>5.1f}%")
print("=" * 62)


  BANGALORE TECH SALARY DECODER
  Built by [Anushka Asawa] | The Unlox Academy | 2-Hour Live Project
  Dataset : 1000 Bengaluru tech professionals
  Period  : 2024 employment snapshot

  ---- MEDIAN CTC BY ROLE (in LPA) ----
  Product Manager    ████████████████████    31.3 LPA
  Data Scientist     ████████████████        24.7 LPA
  SDE Full-Stack     ██████████████          22.4 LPA
  DevOps Engineer    ██████████████          22.1 LPA
  SDE Frontend       ██████████████          21.2 LPA
  SDE Backend        ██████████████          21.1 LPA
  UI/UX Designer     ████████████            18.9 LPA
  Business Analyst   ████████████            18.3 LPA
  Data Analyst       ███████████             16.9 LPA

  ---- SDE BACKEND EXPERIENCE TRAJECTORY ----
  0-1 Years Exp (Entry)      :  11.6 LPA
  2-3 Years Exp (Junior)     :  20.0 LPA
  4-5 Years Exp (Mid-Level)  :  25.9 LPA
  6+  Years Exp (Senior)     :  40.4 LPA

  ---- SYSTEM DESIGN SKILL PREMIUM FOR SDEs ----
  Engineers with System Desi

**Key Insights**

1.(Unicorn Premium): Unicorns pay a median of 27.4 LPA for SDE Backend talent, which is 33.7% more than MNCs (20.5 LPA).Prioritize Unicorn startups over legacy MNCs during job hunts to maximize initial pay.

2 .(Skill Premium): SDEs with System Design on their profile command a median of 25.3 LPA vs 21.0 LPA without it (a +4.3 LPA premium). Master system architecture concepts to break past the 25 LPA salary threshold.

3.(Experience Jump): Mid-level SDE Backend devs (2-3 years exp) earn 20.0 LPA, a huge 71.7% hike over entry-level freshers (11.7 LPA).Focus on staying stable and building core skills in your first 2 years instead of chasing early lateral switches.